# Diverse Models + OOF Probability Blender

## tl;dr

Train independent model families, create leakage-safe OOF probabilities, tune blend weights and class thresholds for balanced accuracy, then export Kaggle submissions.

## Key assumptions

- Competition metric: balanced accuracy.
- `train.csv` and `test.csv` are available in Kaggle input or local `../dataset`.
- CatBoost, scikit-learn, and optionally XGBoost are available on Kaggle.


## 1. Setup and data


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import OrdinalEncoder

SEED = 42
N_SPLITS = 3
TARGET = 'health_condition'
ID = 'id'
LABELS = ['fit', 'at-risk', 'unhealthy']
OUTPUT = Path('/kaggle/working/diverse_blends') if Path('/kaggle/working').exists() else Path('diverse_blends')

def find_train():
    candidates = [Path('../dataset/train.csv'), Path('dataset/train.csv')]
    root = Path('/kaggle/input')
    if root.exists():
        candidates += list(root.rglob('train.csv'))
    for path in candidates:
        if path.exists() and TARGET in pd.read_csv(path, nrows=0).columns:
            return path
    raise FileNotFoundError('train.csv with health_condition not found')

train_path = find_train()
test_path = train_path.parent / 'test.csv'
if not test_path.exists():
    raise FileNotFoundError(f'test.csv not found beside {train_path}')
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
print(train_path, train.shape, test.shape)
print(train[TARGET].value_counts(normalize=True).reindex(LABELS))


## 2. Compact feature engineering


In [ ]:
NUMERIC = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake']
CATEGORICAL = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

def features(frame):
    df = frame.drop(columns=[TARGET], errors='ignore').copy()
    for col in NUMERIC + CATEGORICAL:
        if col in df:
            df[f'{col}_missing'] = df[col].isna().astype('int8')
    df['sleep_deprived'] = (df['sleep_duration'] < 6).astype('int8')
    df['long_sleep'] = (df['sleep_duration'] >= 8).astype('int8')
    df['exercise_per_1k_steps'] = df['exercise_duration'] / (df['step_count'] / 1000).replace(0, np.nan)
    df['calories_per_1k_steps'] = df['calorie_expenditure'] / (df['step_count'] / 1000).replace(0, np.nan)
    df['sleep_water'] = df['sleep_duration'] * df['water_intake']
    for left, right, name in [('stress_level', 'physical_activity_level', 'stress_activity'), ('stress_level', 'sleep_quality', 'stress_sleep'), ('diet_type', 'smoking_alcohol', 'diet_smoking')]:
        df[name] = df[left].fillna('Missing').astype(str) + '|' + df[right].fillna('Missing').astype(str)
    return df

X = features(train)
X_test = features(test)
y = train[TARGET].to_numpy()
print('features:', X.shape[1])


## 3. OOF probabilities from diverse model families


In [ ]:
def encode_for_numeric(train_frame, valid_frame, test_frame):
    categorical = train_frame.select_dtypes(include=['object', 'string']).columns.tolist()
    numeric = [c for c in train_frame.columns if c not in categorical]
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_out = train_frame.copy(); valid_out = valid_frame.copy(); test_out = test_frame.copy()
    if categorical:
        encoder.fit(train_out[categorical].fillna('Missing').astype(str))
        for frame in [train_out, valid_out, test_out]:
            frame[categorical] = encoder.transform(frame[categorical].fillna('Missing').astype(str))
    for col in numeric:
        median = train_out[col].median()
        for frame in [train_out, valid_out, test_out]:
            frame[col] = frame[col].replace([np.inf, -np.inf], np.nan).fillna(median)
    return train_out, valid_out, test_out

def sample_weights(y_part):
    counts = pd.Series(y_part).value_counts()
    return np.array([1.0 / counts[label] for label in y_part]) * len(y_part) / len(counts)

def fit_predict(model_name, train_part, y_part, valid_part, y_valid, test_part):
    if model_name == 'catboost':
        from catboost import CatBoostClassifier
        cats = train_part.select_dtypes(include=['object', 'string']).columns.tolist()
        train_part = train_part.copy(); valid_part = valid_part.copy(); test_part = test_part.copy()
        for frame in [train_part, valid_part, test_part]:
            for col in cats:
                frame[col] = frame[col].fillna('Missing').astype(str)
        model = CatBoostClassifier(iterations=1200, depth=7, learning_rate=0.04, loss_function='MultiClass', eval_metric='MultiClass', class_names=LABELS, auto_class_weights='Balanced', random_seed=SEED, verbose=False, allow_writing_files=False, thread_count=-1)
        model.fit(train_part, y_part, cat_features=cats, eval_set=(valid_part, y_valid), early_stopping_rounds=80)
    elif model_name == 'histgb':
        from sklearn.ensemble import HistGradientBoostingClassifier
        train_part, valid_part, test_part = encode_for_numeric(train_part, valid_part, test_part)
        y_part = np.array([label_to_index[label] for label in y_part])
        model = HistGradientBoostingClassifier(max_iter=700, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=40, l2_regularization=1.0, random_state=SEED)
        model.fit(train_part, y_part, sample_weight=sample_weights(y_part))
    elif model_name == 'xgboost':
        from xgboost import XGBClassifier
        train_part, valid_part, test_part = encode_for_numeric(train_part, valid_part, test_part)
        model = XGBClassifier(n_estimators=900, max_depth=8, learning_rate=0.04, subsample=0.85, colsample_bytree=0.85, objective='multi:softprob', num_class=len(LABELS), eval_metric='mlogloss', tree_method='hist', random_state=SEED, n_jobs=-1)
        y_part = np.array([label_to_index[label] for label in y_part])
        model.fit(train_part, y_part, sample_weight=sample_weights(y_part), verbose=False)
    else:
        raise ValueError(model_name)
    return model.predict_proba(valid_part), model.predict_proba(test_part)

available_models = ['histgb']
try:
    import catboost
    available_models.insert(0, 'catboost')
except ImportError:
    print('CatBoost unavailable; skipped')
try:
    import xgboost
    available_models.append('xgboost')
except ImportError:
    print('XGBoost unavailable; skipped')

label_to_index = {label: i for i, label in enumerate(LABELS)}
y_index = np.array([label_to_index[label] for label in y])
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof = {name: np.zeros((len(X), len(LABELS)), dtype='float32') for name in available_models}
test_probabilities = {name: np.zeros((len(X_test), len(LABELS)), dtype='float32') for name in available_models}

for fold, (fit_idx, valid_idx) in enumerate(splitter.split(X, y_index)):
    print('fold', fold)
    for name in available_models:
        valid_proba, test_proba = fit_predict(name, X.iloc[fit_idx], y[fit_idx], X.iloc[valid_idx], y[valid_idx], X_test)
        oof[name][valid_idx] = valid_proba
        test_probabilities[name] += test_proba / N_SPLITS

model_scores = pd.DataFrame([{'model': name, 'oof_balanced_accuracy': balanced_accuracy_score(y_index, oof[name].argmax(axis=1))} for name in available_models]).sort_values('oof_balanced_accuracy', ascending=False)
display(model_scores)


## 4. OOF blend weight search


In [ ]:
def best_weights(names, probabilities, y_true):
    if len(names) == 1:
        return np.array([1.0])
    best_score, best = -1.0, None
    grid = np.arange(0, 1.01, 0.10)
    for a in grid:
        for b in grid:
            c = 1.0 - a - b
            if len(names) == 3 and c >= 0:
                weights = np.array([a, b, c])
            elif len(names) == 2 and abs(b - (1 - a)) < 1e-9:
                weights = np.array([a, b])
            else:
                continue
            blended = sum(w * probabilities[n] for w, n in zip(weights, names))
            score = balanced_accuracy_score(y_true, blended.argmax(axis=1))
            if score > best_score:
                best_score, best = score, weights
    return best

blend_names = available_models
selection_split = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
selection_idx, holdout_idx = next(selection_split.split(np.zeros(len(y_index)), y_index))
selection_probabilities = {name: probabilities[selection_idx] for name, probabilities in oof.items()}
selection_y = y_index[selection_idx]
weights = best_weights(blend_names, selection_probabilities, selection_y)
oof_blend = sum(weight * oof[name] for weight, name in zip(weights, blend_names))
test_blend = sum(weight * test_probabilities[name] for weight, name in zip(weights, blend_names))
print(dict(zip(blend_names, weights)))
print('selection blend balanced accuracy:', balanced_accuracy_score(selection_y, oof_blend[selection_idx].argmax(axis=1)))
print('holdout blend balanced accuracy:', balanced_accuracy_score(y_index[holdout_idx], oof_blend[holdout_idx].argmax(axis=1)))
print('holdout model scores:')
for name in blend_names:
    print(name, balanced_accuracy_score(y_index[holdout_idx], oof[name][holdout_idx].argmax(axis=1)))


## 5. Class-specific threshold tuning


In [ ]:
def apply_thresholds(probabilities, thresholds):
    return (probabilities / np.asarray(thresholds)).argmax(axis=1)

def tune_thresholds(probabilities, y_true):
    thresholds = np.ones(len(LABELS))
    grid = np.arange(0.70, 1.31, 0.05)
    for _ in range(2):
        for class_index in range(len(LABELS)):
            scores = []
            for value in grid:
                candidate = thresholds.copy(); candidate[class_index] = value
                scores.append((balanced_accuracy_score(y_true, apply_thresholds(probabilities, candidate)), value))
            thresholds[class_index] = max(scores)[1]
    return thresholds

thresholds = tune_thresholds(oof_blend[selection_idx], selection_y)
plain_pred = oof_blend[holdout_idx].argmax(axis=1)
tuned_pred = apply_thresholds(oof_blend[holdout_idx], thresholds)
print('thresholds:', dict(zip(LABELS, thresholds)))
print('holdout plain:', balanced_accuracy_score(y_index[holdout_idx], plain_pred))
print('holdout tuned:', balanced_accuracy_score(y_index[holdout_idx], tuned_pred))


## 6. Export submissions


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
def save_submission(name, probabilities, thresholds=None):
    pred = apply_thresholds(probabilities, thresholds) if thresholds is not None else probabilities.argmax(axis=1)
    frame = pd.DataFrame({ID: test[ID], TARGET: np.array(LABELS)[pred]})
    path = OUTPUT / f'{name}.csv'
    frame.to_csv(path, index=False)
    assert frame.columns.tolist() == [ID, TARGET]
    assert frame[ID].is_unique and len(frame) == len(test)
    return path

paths = {name: save_submission(f'submission_{name}', test_probabilities[name]) for name in available_models}
paths['blend_plain'] = save_submission('submission_blend_plain', test_blend)
paths['blend_tuned'] = save_submission('submission_blend_tuned', test_blend, thresholds)
print('saved:')
for name, path in paths.items():
    print(name, path)


## Takeaways

Compare `plain` versus `tuned` OOF score. Submit tuned thresholds only if the gain survives a separate holdout or repeated folds. Keep the best existing public anchor as a control submission.
